[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-03-evaluations.ipynb#scrollTo=ba01ca02)

---
# Day 3 · Prompt Evaluation Systems
**certified-journeys / claude-api-certified** · Test datasets, grading & eval workflows

> **Goal for today:** Build a repeatable eval pipeline — test dataset, grader function, A/B comparison, and Claude-as-judge — so you can improve prompts with data instead of intuition.

In [ ]:
%pip install -q anthropic pandas

In [ ]:
import os
import json
import pandas as pd
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## Step 1 · Build a test dataset

A good eval dataset has:
- **Inputs** that cover common and edge cases
- **Expected outputs** or criteria for correct answers
- **Diversity** — don't test only the easy cases

| Dataset type | When to use |
|---|---|
| Exact match | Classification, extraction, yes/no questions |
| Criteria-based | Open-ended generation, summarization |
| Claude-as-judge | Complex quality assessment at scale |

In [ ]:
# Sentiment classification dataset
EVAL_DATASET = [
    {"input": "I love this product! It works perfectly.", "expected": "positive"},
    {"input": "Terrible experience, would not recommend.", "expected": "negative"},
    {"input": "It's okay, nothing special.", "expected": "neutral"},
    {"input": "Absolutely amazing, exceeded all expectations!", "expected": "positive"},
    {"input": "Broke after two days of use. Very disappointed.", "expected": "negative"},
    {"input": "Decent quality for the price.", "expected": "neutral"},
    {"input": "Five stars! Best purchase of the year.", "expected": "positive"},
    {"input": "Not what was advertised. Very misleading.", "expected": "negative"},
    {"input": "Average product, does what it says.", "expected": "neutral"},
    {"input": "Life-changing! I recommend to everyone.", "expected": "positive"},
]

df = pd.DataFrame(EVAL_DATASET)
print(f"Dataset size: {len(df)} examples")
print(df.groupby("expected").size().rename("count"))

**What just happened?**
- A balanced dataset with 10 examples across 3 classes.
- **Edge cases matter most** — if your prompt fails on neutral/ambiguous inputs, that's where to focus.
- **Save datasets as CSV** so evals are reproducible across prompt versions.

## Step 2 · Write a grading function

A grader takes a model output and expected output and returns a score. Start simple (exact match), then move to semantic matching when needed.

In [ ]:
PROMPT_V1 = "Classify the sentiment of this text as exactly one word: positive, negative, or neutral.\n\nText: {input}"

def run_prompt(prompt_template: str, input_text: str) -> str:
    """Run a prompt template and return the cleaned response."""
    r = client.messages.create(
        model=MODEL,
        max_tokens=16,
        temperature=0,  # deterministic for classification
        messages=[{"role": "user", "content": prompt_template.format(input=input_text)}]
    )
    return r.content[0].text.strip().lower()


def grade_exact(prediction: str, expected: str) -> float:
    """1.0 if exact match, 0.0 otherwise."""
    return 1.0 if prediction == expected else 0.0


def run_eval(prompt_template: str, dataset: list[dict]) -> pd.DataFrame:
    """Run a prompt against a dataset and return scored results."""
    results = []
    for example in dataset:
        prediction = run_prompt(prompt_template, example["input"])
        score = grade_exact(prediction, example["expected"])
        results.append({
            "input": example["input"][:40] + "...",
            "expected": example["expected"],
            "predicted": prediction,
            "score": score
        })
    return pd.DataFrame(results)


print("Running eval for Prompt V1...")
results_v1 = run_eval(PROMPT_V1, EVAL_DATASET)
print(results_v1.to_string(index=False))
print(f"\nAccuracy: {results_v1['score'].mean():.1%}")

**What just happened?**
- `temperature=0` ensures the same input always produces the same output — essential for reproducible evals.
- **Exact match grading** works for classification tasks where the output space is small and well-defined.
- Look at the failures — they reveal where your prompt is ambiguous or incomplete.
- **Never optimize a prompt based on a single example** — always eval on the full dataset.

## Step 3 · A/B prompt comparison

Run two prompt variants against the same dataset and compare. This is how you make data-driven decisions about which prompt to ship.

In [ ]:
PROMPT_V2 = """You are a sentiment classifier. Analyze the customer review below and respond with exactly one of these labels: positive, negative, or neutral.

Rules:
- positive: overall pleased, satisfied, or enthusiastic
- negative: overall displeased, frustrated, or disappointed
- neutral: mixed, indifferent, or neither clearly positive nor negative

Respond with only the label. No explanation.

Review: {input}"""

print("Running eval for Prompt V2...")
results_v2 = run_eval(PROMPT_V2, EVAL_DATASET)

acc_v1 = results_v1["score"].mean()
acc_v2 = results_v2["score"].mean()

print(f"\n{'Prompt':<12} {'Accuracy':>10} {'Correct':>10}")
print("-" * 34)
print(f"{'V1':<12} {acc_v1:>9.1%} {int(acc_v1 * len(EVAL_DATASET)):>10}/{len(EVAL_DATASET)}")
print(f"{'V2':<12} {acc_v2:>9.1%} {int(acc_v2 * len(EVAL_DATASET)):>10}/{len(EVAL_DATASET)}")
print(f"\nWinner: {'V2' if acc_v2 >= acc_v1 else 'V1'} (+{abs(acc_v2 - acc_v1):.1%} improvement)")

**What just happened?**
- V2 added explicit label definitions, which typically reduces ambiguity for edge cases.
- **Even a 10% improvement on 10 examples is meaningful signal** — scale to 100+ examples for statistical confidence.
- Always save your eval results to CSV with a timestamp so you can track prompt history over time.
- When accuracy plateaus, look at where both versions fail — those are your hardest cases.

## Step 4 · Claude-as-judge for open-ended outputs

Exact match doesn't work for summarization, code generation, or creative tasks. Use Claude itself as a judge — it's fast, scalable, and surprisingly reliable.

In [ ]:
JUDGE_SYSTEM = """You are an objective AI evaluator. You will be given a task, an AI response, and evaluation criteria.
Score the response from 1-5 on each criterion, then give an overall score.
Respond in JSON only: {"criteria_scores": {"criterion": score}, "overall": score, "reasoning": "one sentence"}"""

def judge_response(
    task: str,
    response: str,
    criteria: list[str]
) -> dict:
    """Use Claude to score an AI response against given criteria."""
    criteria_str = "\n".join(f"- {c}" for c in criteria)
    prompt = f"""Task given to AI: {task}

AI Response: {response}

Evaluation criteria (score 1-5 each):
{criteria_str}"""

    r = client.messages.create(
        model=MODEL,
        max_tokens=256,
        temperature=0,
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        return json.loads(r.content[0].text)
    except json.JSONDecodeError:
        return {"error": "invalid JSON", "raw": r.content[0].text}


# Test: evaluate a summary
task = "Summarize what machine learning is in 2 sentences for a non-technical audience."
r = client.messages.create(
    model=MODEL, max_tokens=128,
    messages=[{"role": "user", "content": task}]
)
candidate_response = r.content[0].text

judgment = judge_response(
    task=task,
    response=candidate_response,
    criteria=["Accuracy", "Clarity for non-technical audience", "Conciseness"]
)

print("Response to evaluate:")
print(candidate_response)
print("\nJudgment:")
print(json.dumps(judgment, indent=2))

**What just happened?**
- Claude-as-judge replaces human annotation for many quality dimensions — at a fraction of the cost and time.
- **JSON output + `temperature=0`** makes the judge reliable and parseable.
- **Bias note:** Claude-as-judge can favor Claude outputs. For critical evals, use a different model as judge (e.g., GPT-4) or human spot-checks.
- Always include `"reasoning"` in the judge output so you can audit failures.

## Step 5 · Build a full eval report

Combine all the pieces into a reusable eval runner that produces a summary report.

In [ ]:
def eval_report(prompt_name: str, results_df: pd.DataFrame) -> None:
    """Print a formatted eval summary."""
    total = len(results_df)
    correct = results_df["score"].sum()
    accuracy = results_df["score"].mean()
    failures = results_df[results_df["score"] < 1.0]

    print(f"\n{'='*50}")
    print(f"Eval Report: {prompt_name}")
    print(f"{'='*50}")
    print(f"Total examples : {total}")
    print(f"Correct        : {int(correct)}/{total}")
    print(f"Accuracy       : {accuracy:.1%}")
    if not failures.empty:
        print(f"\nFailures ({len(failures)}):")
        for _, row in failures.iterrows():
            print(f"  Input: {row['input']}")
            print(f"  Expected: {row['expected']} | Got: {row['predicted']}")


eval_report("Prompt V1", results_v1)
eval_report("Prompt V2", results_v2)

**What just happened?**
- A complete eval report shows overall accuracy plus a detailed failure analysis.
- **Failure analysis is where the improvement lives** — look at patterns in what fails.
- Save `results_df` to CSV (`results_df.to_csv('eval_v1_2025-01-01.csv', index=False)`) for historical tracking.
- For production, run evals on every prompt change as part of your CI pipeline.

In [ ]:
# Challenge: Write a better grading function
# The current grade_exact() only returns 0 or 1.
# Write grade_lenient() that returns 0.5 for common near-misses.

NEAR_MISSES = {
    # (predicted, expected) -> partial credit
    ("very positive", "positive"): 0.5,
    ("very negative", "negative"): 0.5,
    ("mixed", "neutral"): 0.5,
}

def grade_lenient(prediction: str, expected: str) -> float:
    """Return 1.0 for exact match, 0.5 for near-miss, 0.0 otherwise."""
    # Your solution here
    pass


assert grade_lenient("positive", "positive") == 1.0, "exact match should return 1.0"
assert grade_lenient("very positive", "positive") == 0.5, "near-miss should return 0.5"
assert grade_lenient("negative", "positive") == 0.0, "wrong answer should return 0.0"
print("✓ Challenge passed!")

---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Test dataset | Cover common + edge cases; balanced across classes |
| Exact match grader | Best for classification with small output space |
| A/B comparison | Run both prompts on same dataset; compare accuracy |
| Claude-as-judge | Fast, scalable quality scoring for open-ended tasks |
| `temperature=0` | Always use for grading functions — must be deterministic |

> **Tip:** Evaluation is the difference between vibe-coding and AI engineering. Build your eval suite before you optimize prompts.

---
## What's next
**Day 4** → Prompt Engineering Techniques: XML tags, chain-of-thought, few-shot examples, and the Anthropic prompt library.

Mark Day 3 complete in your [tracker](../index.html).